# 04 · Hyperparameter Tuning

**When to run this notebook:** once you are happy with the model set from
notebook 03 and want to optimize the **top** models. You explicitly name which
models to tune — tuning every model on every run is expensive and rarely
worthwhile.

## Approach

`RandomizedSearchCV` (configurable strategy / iterations / folds, ROC-AUC
scoring) searches the named models only; models left out keep their current
snapshot parameters. All four families are still trained and evaluated so the
comparison stays complete — only the **search** is restricted to the named
subset.

> **Sequence:** notebook 4 of 5. See [notebooks.md](./notebooks.md).

## Configuration

- **`IS_DRY_RUN`** (default `True`) — no hyperparameter snapshot, model artifacts,
  or version bump are written.
- **`MODELS_TO_TUNE`** — the explicit subset to search. Accepts friendly names
  (`"xgb"`, `"rf"`, `"lr"`) or class names. Set to e.g. `["xgb"]` to tune only
  XGBoost.

In [1]:
import sys
import os

# Make the capstone package importable (notebook lives in notebooks/, code in src/capstone/).
sys.path.append(os.path.abspath("../"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory
from pipeline.stages import eda
from pprint import pprint
from utils.tune_hyperparameters import get_all_default_param_grids

IMAGES_FOLDER = "../../../images/results/"

# This notebook produces no figures; PERSIST_DATA stays False so a re-run writes
# NO data — it shows the search results in-notebook without persisting tuned
# hyperparameters/models or bumping any version. Flip to True to persist.
PERSIST_DATA = False

# Explicit subset of models to search (the "top" models). The rest keep their
# loaded snapshot params. Friendly aliases or class names both work.
# tune the three independent base models; ensembles inherit automatically
MODELS_TO_TUNE = ["xgb", "lgb", "mlp"]

# Metric column order used in every results table below. Global (macro) metrics
# come first, then the per-class above/below breakdown.
#   *_macro = sklearn average="macro" = unweighted mean of above- and below-baseline
#             scores. It is the single "overall" number that weights both classes equally.
METRIC_COLS = [
    "roc_auc", "accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_above", "recall_above", "f1_above",
    "precision_below", "recall_below", "f1_below",
]

### Search grids

Start from the project defaults, then override per model as needed. Only the
grids for models in `MODELS_TO_TUNE` are actually used.

> **MLP note:** `early_stopping` is hardcoded to `True` in `ModelTrainer` and
> is stripped from the params dict before the classifier is constructed. Do not
> add it to the MLP grid — it will be saved into the snapshot and cause a
> duplicate-keyword error on the next training run.

In [2]:
new_grids = get_all_default_param_grids()
pprint(new_grids)

{'LGBMClassifier': {'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
                    'learning_rate': [0.01, 0.05, 0.1, 0.2],
                    'max_depth': [-1, 5, 8, 12],
                    'min_child_samples': [5, 10, 20, 50],
                    'n_estimators': [100, 200, 300, 500],
                    'num_leaves': [15, 31, 63, 127],
                    'subsample': [0.6, 0.7, 0.8, 1.0]},
 'LogisticRegression': {'C': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 100.0],
                        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
                        'max_iter': [2000, 5000],
                        'penalty': ['l1', 'elasticnet'],
                        'solver': ['saga']},
 'MLPClassifier': {'activation': ['relu', 'tanh'],
                   'alpha': [0.0001, 0.001, 0.01, 0.1],
                   'hidden_layer_sizes': [(64,),
                                          (128,),
                                          (64, 64),
                                          (128, 64),


### Tuning config

`tune(..., models=MODELS_TO_TUNE)` restricts the search to the named subset.
`snapshot_hyperparams()` and `snapshot_models()` stage the writes (gated by
`IS_DRY_RUN`).

In [3]:
# snapshot_hyperparams() and snapshot_models() are chained unconditionally — dry_run()
# decides whether they take effect, so no version bump appears during a dry run.
config = (
    VersionConfig.load(use_synthetic=False)
    .tune(strategy="random", n_iter=100, cv=5, new_grids=new_grids, models=MODELS_TO_TUNE)
    .snapshot_hyperparams()
    .snapshot_models()
    .dry_run(not PERSIST_DATA)
    .build()
)

run = PipelineRun(config)
stages = PipelineFactory.tune_hyperparams(config)
print("Scenario:", stages.scenario)
print("Tuning models:", config.tune_model_names)

VersionConfig loaded:
  data:          v3.5 (raw_suffix='real')
  baselines:     v5.0
  model:         v6.1
  hyperparams:   v1.1
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['models', 'hyperparams', 'tune']
  data              : v3.5 (unchanged)
  baselines         : v5.0 (unchanged)
  model             : v6.1 -> v6.2
  hyperparams       : v1.1 -> v1.2
  raw_version       : v3.5_real
  next_final_version: v3.5_100real
  model_version     : v6.1  ->  v6.2
  baselines_version : v5.0
  hyperparam_version: v1.1  ->  v1.2
  use_synthetic     : False
  dry_run           : False
  Tuning            : strategy=random, n_iter=100, cv=5, scoring=roc_auc
  Tuning models     : ['XGBoost', 'LightGBM', 'MLP']

  Call config.commit() after all snapshots succeed.
Scenario: tune_hyperparams
Tuning models: ['XGBoost', 'LightGBM', 'MLP']


### Load, process, split & scale

`DataLoader` is the expensive step, so it is run on its own; the rest of the
pre-modeling stages follow.

In [4]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.5_real': 106626 rows from 2026-05-29
  Polls: {'upload': 37387, '24h': 36506, '7d': 32733}
Loaded baselines 'v5.0': 40841 baseline videos, 1375 median rows (1375 channels)
PipelineRun(config=VersionConfig(data=(3, 5), model=(6, 2), baselines=(5, 0), hyperparams=(1, 2), use_synthetic=False, active=['models', 'hyperparams', 'tune']))
  df_videos      populated  DataFrame shape=(106626, 23)
  df_baselines   populated  DataFrame shape=(40841, 15)
  df_medians     populated  DataFrame shape=(1375, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  df_gen         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  X_gen          None
  y_train        None
  y_test         None
  y_val          None
  y_gen          None
  models         empty dict
  results        empty dict


In [5]:
stages.preprocessor.run(run)
stages.engineer.run(run)
stages.splitter.run(run)
stages.scaler.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 32571 (dropped 4864 incomplete)
  Pivoted shape: (32740, 34)

[2/3] Joining baseline medians...
  Baseline join: 32740/32740 videos matched a channel median

[3/3] Cleaning data...
  Dropped 15 row(s) with null contains_synthetic_media (private/failed harvest).
  Cleaned: 32725 rows × 40 columns

Clean dataset: 32725 rows × 40 columns
  all: dropped 599 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/10] Computing target 

### Search + train

With tuning enabled, `ModelTrainer` runs `RandomizedSearchCV` (5-fold CV) on the
named models, then fits all four final models. Models outside `MODELS_TO_TUNE`
are fit with their loaded snapshot params.

In [6]:
stages.trainer.run(run)
if run.tune_elapsed_s is not None:
    mins, secs = divmod(run.tune_elapsed_s, 60)
    print(f"\nTuning wall time: {run.tune_elapsed_s:.1f}s ({int(mins)}m {secs:.0f}s)")
run.summary()

Loaded hyperparams 'v1.1' (saved 2026-04-29)
  Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier']
  Search: {'strategy': 'random', 'n_iter': 100, 'cv': 5, 'scoring': 'roc_auc'}
Loaded hyperparams from snapshot 'v1.1'.
  Note: injected l1_ratio=0.5 for elasticnet LR (missing from snapshot).
  Note: 'LightGBM' not in snapshot — using defaults.
  Note: 'MLP' not in snapshot — using defaults.
Tuning subset: ['XGBoost', 'LightGBM', 'MLP'] (keeping loaded params for ['LogisticRegression', 'RandomForest'])

Using param_grid: {'n_estimators': [100, 200, 300, 500], 'max_depth': [3, 4, 5, 6, 8], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'subsample': [0.6, 0.7, 0.8, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 'min_child_weight': [1, 3, 5, 10], 'gamma': [0, 0.1, 0.3, 0.5]}
Tuning XGBClassifier with strategy='random' (cv=5, scoring='roc_auc')
  Sampling 100 combinations from grid of ~20,480 total
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best roc_

Exception ignored in: <function ResourceTracker.__del__ at 0x118b19300>
Traceback (most recent call last):
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x104a39300>
Traceback (most recent call last):
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
Chi


=== ModelTrainer — test-set results ===
  lr_l1               AUC=0.7792  acc=0.7129  F1↑=0.7479
  rf                  AUC=0.8922  acc=0.8122  F1↑=0.8384
  xgb                 AUC=0.9271  acc=0.8495  F1↑=0.8653
  lgb                 AUC=0.9234  acc=0.8471  F1↑=0.8640
  mlp                 AUC=0.8651  acc=0.7889  F1↑=0.8123
  ensemble_voting     AUC=0.9243  acc=0.8471  F1↑=0.8641
  ensemble_stacking   AUC=0.9268  acc=0.8506  F1↑=0.8661

Tuning wall time: 1596.6s (26m 37s)
PipelineRun(config=VersionConfig(data=(3, 5), model=(6, 2), baselines=(5, 0), hyperparams=(1, 2), use_synthetic=False, active=['models', 'hyperparams', 'tune']))
  df_videos      populated  DataFrame shape=(106626, 23)
  df_baselines   populated  DataFrame shape=(40841, 15)
  df_medians     populated  DataFrame shape=(1375, 7)
  df_clean       populated  DataFrame shape=(32725, 40)
  df_engineered  populated  DataFrame shape=(32126, 89)
  df_train       populated  DataFrame shape=(19589, 89)
  df_test        populated

### Persist tuned hyperparameters + models

Gated by `IS_DRY_RUN`. With `IS_DRY_RUN=False` these write the new hyperparameter
version and the retrained model artifacts to GCS.

In [ ]:
stages.hyperparam_snapshotter.run(run)
stages.model_snapshotter.run(run)

Model artifacts saved to models/v6.2_lr_l1/
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_lr_l1/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_lr_l1/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_lr_l1/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_lr_l1/metadata.json

Model v6.2_lr_l1 (LogisticRegression)
  Data: v3.5_100real (19589 real + 0 synthetic)
  ROC-AUC: 0.7792
  Accuracy: 0.7129
  F1 (above): 0.7479
Model artifacts saved to models/v6.2_rf/
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_rf/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_rf/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_rf/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v6.2_rf/metadata.json

Model v6.2_rf (RandomForest)
  Data: v3.5_100real (19589 real + 0 synthetic)
  ROC-AUC: 0.8922
  Accuracy: 0.8122
  F1 (above): 0.8384
Model artifacts saved to models/v6.2_xgb/
  

PipelineRun(model_version='v6.1', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'df_gen', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'X_gen', 'y_train', 'y_test', 'y_val', 'y_gen', 'models'])

### Validate on the locked holdout

In [11]:
stages.validator.run(run)
stages.validation_results_snapshotter.run(run)


=== Validator — validation-set results ===
  lr_l1           AUC=0.7679  acc=0.7036  F1↑=0.7384
  rf              AUC=0.8861  acc=0.8062  F1↑=0.8332
  xgb             AUC=0.9203  acc=0.8450  F1↑=0.8621
  lgb             AUC=0.9188  acc=0.8442  F1↑=0.8619
  mlp             AUC=0.8503  acc=0.7735  F1↑=0.7972
  ensemble_voting  AUC=0.9173  acc=0.8428  F1↑=0.8608
  ensemble_stacking  AUC=0.9204  acc=0.8450  F1↑=0.8621
Validation results appended → gs://maduros-dolce-capstone-data/models/v6.2/validation_results.jsonl
  lr_l1           AUC=0.7679  acc=0.7036  F1↑=0.7384
  rf              AUC=0.8861  acc=0.8062  F1↑=0.8332
  xgb             AUC=0.9203  acc=0.8450  F1↑=0.8621
  lgb             AUC=0.9188  acc=0.8442  F1↑=0.8619
  mlp             AUC=0.8503  acc=0.7735  F1↑=0.7972
  ensemble_voting  AUC=0.9173  acc=0.8428  F1↑=0.8608
  ensemble_stacking  AUC=0.9204  acc=0.8450  F1↑=0.8621


PipelineRun(model_version='v6.1', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'df_gen', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'X_gen', 'y_train', 'y_test', 'y_val', 'y_gen', 'models', 'results'])

### A note on the evaluation metrics

Every metrics table in this notebook leads with three **global** scores —
`precision_macro`, `recall_macro`, `f1_macro` — followed by the per-class
breakdown (`*_above`, `*_below`).

The global scores use **sklearn's `average="macro"`**: compute precision /
recall / F1 separately for each class, then take the unweighted mean across
classes. For example, `f1_macro = mean(f1_above, f1_below)`. Because the
target is ~50/50 by design, macro and weighted averages are nearly identical
here — macro is preferred because it treats both classes equally regardless
of exact counts, which matches the intent of a balanced binary target.

The per-class split is retained because the error costs differ — a false
"will overperform" prediction has a different business impact than a false
"will underperform" — but the macro scores are shown first as the headline
numbers.

In [12]:
df_val = pd.DataFrame(run.results).T[METRIC_COLS].astype(float).round(4)
df_val.index.name = "model"
df_val.style.highlight_max(color="#d4edda").format("{:.4f}")

,roc_auc,accuracy,precision_macro,recall_macro,f1_macro,precision_above,recall_above,f1_above,precision_below,recall_below,f1_below
model,,,,,,,,,,,
lr_l1,0.7679,0.7036,0.6998,0.6974,0.6982,0.7242,0.7531,0.7384,0.6753,0.6417,0.6581
rf,0.8861,0.8062,0.8083,0.7982,0.8011,0.7985,0.8710,0.8332,0.8182,0.7253,0.7689
xgb,0.9203,0.8450,0.8438,0.8415,0.8425,0.8520,0.8724,0.8621,0.8356,0.8107,0.8230
lgb,0.9188,0.8442,0.8434,0.8403,0.8416,0.8490,0.8752,0.8619,0.8378,0.8055,0.8213
mlp,0.8503,0.7735,0.7708,0.7700,0.7703,0.7929,0.8017,0.7972,0.7487,0.7383,0.7435
ensemble_voting,0.9173,0.8428,0.8421,0.8388,0.8402,0.8472,0.8748,0.8608,0.8369,0.8029,0.8195
ensemble_stacking,0.9204,0.8450,0.8438,0.8415,0.8425,0.8520,0.8724,0.8621,0.8356,0.8107,0.8230


### Best parameters found

The search results for the tuned models. Compare these against the prior
snapshot to confirm the search moved in a sensible direction (e.g. more
estimators, stronger regularization) and that validation AUC improved or held.

In [13]:
for name, entry in run.models.items():
    cfg = entry["model_config"]
    print(f"{name}  ({cfg.model_type}):")
    pprint(cfg.hyperparameters)
    print()

lr_l1  (LogisticRegression):
{'C': 10.0, 'max_iter': 5000, 'penalty': 'elasticnet', 'solver': 'saga'}

rf  (RandomForest):
{'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 800}

xgb  (XGBoost):
{'colsample_bytree': 1.0,
 'learning_rate': 0.05,
 'max_depth': 8,
 'min_child_weight': 5,
 'n_estimators': 500,
 'subsample': 0.8}

lgb  (LightGBM):
{'colsample_bytree': 0.7,
 'learning_rate': 0.05,
 'max_depth': -1,
 'min_child_samples': 20,
 'n_estimators': 500,
 'num_leaves': 127,
 'subsample': 1.0}

mlp  (MLP):
{'activation': 'tanh',
 'alpha': 0.001,
 'early_stopping': True,
 'hidden_layer_sizes': [256, 128],
 'learning_rate_init': 0.001,
 'max_iter': 300}

ensemble_voting  (VotingClassifier):
{'estimators': ['rf', 'xgb'], 'voting': 'soft', 'weights': [1, 2]}

ensemble_stacking  (StackingClassifier):
{'cv': 5,
 'estimators': ['rf', 'xgb', 'lgb'],
 'final_estimator': 'LogisticRegression'}



## Findings — tuning

<TODO(jelani): Add findings, touching on topics below>

- which models improved on validation ROC-AUC and by how much
- whether the gains justify the added complexity. 
- Based on prior modeling runs, Tree-model gains here are often incremental; the LR ceiling is structural (see notebook 03).

**Next:** [05 · Final model selection + results](./05_final_model_selection.ipynb).

## Persist version bump (GCS)

Runs only when `IS_DRY_RUN=False`.

In [14]:
config.commit()

Committed versions.json -> data v3.5, model v6.2, baselines v5.0, hyperparams v1.2
